# Revision analyses for "Simulation-based validation of scalp EEG beta-burst imaging in Parkinson's disease"

This notebook adds the analyses requested in review. It **does not change** any result of the two main
notebooks. It imports their code and reads their cached outputs.

**Put this notebook in the same folder as** `pd_beta_three_datasets.ipynb`, `beta_burst_sim.ipynb` and the
`pd_beta_three/` output folder, after the final run of both notebooks.

| § | Analysis | Reviewer point | Needs EEG? | Time (desktop) |
|---|---|---|---|---|
| R0 | Software versions; ico4 ⊂ ico5 overlap check | reproducibility; "never coincided" sentence | no | < 1 min |
| R1 | Simulation with the **real-data detector** (flanking-band GED + gate, 1/f-normalised envelope) and three threshold scopes: per recording, one common threshold per simulated cohort, and the real dataset's threshold. Noise-only false-burst rates | validation used a different detector than the real data | no | ~25 min per dataset |
| R2 | Region-specific BBR → SNR calibration (M1, PMd, SMA) with bootstrap CIs; expected accuracy from R1 | calibration was M1 only | no | ~10 min per dataset |
| R3 | Simulation sensitivities: patch radius, two generators (independent and synchronous), STN, EMG contamination, low/high beta | model assumptions | no | ~30 min |
| R4 | Quality-gate sensitivity (11 gate settings + no gate) for burst metrics, PD vs HC, OFF vs ON and meta-analysis; included vs excluded by group; amplitude decomposition (µV and aperiodic) | gate drives the PD–HC result? normalisation confound? | yes | ~10 min |
| R5 | Extended split-half reliability: burst metrics (ICC), map correlation, refitted vs fixed filter | reliability only of peak location | yes | ~10 min |
| R6 | Surrogates: 200 for envelope CV and duration, 100 for the source maps; FDR | 40 surrogates too few | yes | ~1–2 h (maps) |
| R7 | Random-effects table; bootstrap CIs; minimum detectable effects; TOST equivalence for OFF vs ON | RE and τ² not reported; null results over-interpreted | no | < 1 min |

**Cell order:** R0 → R4–R6 (real EEG; also caches each dataset's real threshold, needed by R1) → R1–R3
(simulation) → R7. Every section saves a CSV in `pd_beta_three/revision/` and prints a compact table.
**Paste back:** the printed output of every cell (or send the `revision/` folder).
Sections can be switched off in the settings cell; every expensive result is cached, so a rerun after a crash
continues where it stopped.

In [1]:
"""Settings."""
from pathlib import Path
import json, ast, gc, time, platform, sys, re, pickle, warnings
import numpy as np
import pandas as pd
from scipy import signal, stats, linalg

MAIN_NOTEBOOK = Path("pd_beta_three_datasets.ipynb")   # the real-data notebook (same folder)
RUN = {"R0": True, "R1": True, "R2": True, "R3": True, "R4": True, "R5": True, "R6": True, "R7": True}
REV_QUICK = False                  # True: tiny smoke test of every section (minutes)

R1_SNRS = (-15.0, -10.0, -5.0, 0.0, 5.0, 10.0)
R1_REGIONS = ["L_M1_hand", "R_M1_hand", "L_PMd", "R_PMd", "L_SMA", "L_occipital", "L_thalamus"]
R1_SEEDS = 10                      # recordings per region × SNR
R1_NOISE_ONLY = 20                 # burst-free recordings per dataset
R1_METHODS = ["dSPM", "eLORETA", "LCMV"]
R2_REGIONS = ["L_M1_hand", "R_M1_hand", "L_PMd", "R_PMd", "L_SMA"]
R2_REPS = 8                        # simulated recordings per region × SNR level
N_BOOT = 2000
R3_DATASET = "sandiego"            # sensitivity simulations use this dataset's montage and duration
R3_SNRS = (-5.0, 0.0)
R3_SEEDS = 10
STN_POS = (-12.0, -14.0, -6.0)     # fsaverage/MNI mm
GATE_GRID = ([{"min_beta_db": b, "max_slope": -0.5, "min_sm_frac": s} for b in (1.0, 2.0, 3.0) for s in (0.3, 0.4, 0.5)]
             + [{"min_beta_db": 2.0, "max_slope": sl, "min_sm_frac": 0.4} for sl in (-0.3, -0.7)])
N_SURR_ENV = 200
N_SURR_MAPS = 100                  # 0 = skip the map surrogates (the slow part of R6)
SESOI_DZ = 0.5                     # equivalence bounds for OFF vs ON (TOST), in d_z

In [2]:
"""Load every definition of the real-data notebook (and through it the simulation notebook) without running any
of its heavy cells: preprocessing, the per-dataset analysis, statistics and figures are skipped."""
SKIP = ("qc_pre, covariates = {}, {}", "RESULTS = {}", "full = {ds: RESULTS", "def _msd", "__import__(_mod)")
nb = json.loads(MAIN_NOTEBOOK.read_text(encoding="utf-8"))
for cell in nb["cells"]:
    src = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
    if cell["cell_type"] == "markdown" and src.lstrip().startswith("# Part E"):
        break
    if cell["cell_type"] != "code" or any(k in src for k in SKIP):
        continue
    exec(compile(src, str(MAIN_NOTEBOOK), "exec"), globals())
warnings.filterwarnings("ignore")
if REV_QUICK:
    R1_SNRS, R1_SEEDS, R1_NOISE_ONLY, R1_REGIONS = (-5.0, 5.0), 2, 2, ["L_M1_hand", "L_SMA", "L_thalamus"]
    R2_REPS, N_BOOT, R3_SEEDS, R3_SNRS = 2, 200, 2, (0.0,)
    GATE_GRID, N_SURR_ENV, N_SURR_MAPS = GATE_GRID[:3], 20, 5
REV = WORK / "revision"
REV.mkdir(parents=True, exist_ok=True)
print(f"Loaded {MAIN_NOTEBOOK}: datasets {RUN_DATASETS}, head model {HEAD_MODEL}, outputs → {REV}")


def save(df, name):
    df.to_csv(REV / f"{name}.csv", index=False)
    return df


def show(t, cols=None, nd=3):
    if t is None or not len(t):
        print("  (no data)")
        return
    t = t[cols] if cols else t
    print(t.round(nd).to_string(index=False))


def boot_ci(x, fn=np.median, n=N_BOOT, seed=0):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    b = [fn(x[rng.integers(0, len(x), len(x))]) for _ in range(n)]
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def per_rec(ds, montage="full", kind="per_recording"):
    f = WORK / "results" / f"{ds}_{montage}_{kind}.csv"
    return pd.read_csv(f) if f.exists() else None


def dataset_channels(ds, montage="full"):
    """The channel set analyse_dataset() used (channels present in every analysed recording)."""
    files = discover_pre(WORK / "preprocessed" / ds / PRIMARY_BLOCK[ds])
    kind = DATASETS[ds]["montage"]
    heads = [set(mne.io.read_raw_eeglab(str(f), preload=False).ch_names) for f in files.file]
    full_ = [c for c in mne.channels.make_standard_montage(montage_name(kind)).ch_names if all(c in h for h in heads)]
    return (full_ if montage == "full" else [c for c in BIOSEMI32 if c in full_]), kind, files


ANAT, DUR, DENSITY_FILE = {}, {}, {}
for ds in RUN_DATASETS:
    r = per_rec(ds, kind="per_recording_no_gate")
    if r is None:
        print(f"{LABEL[ds]}: no results of the main notebook, skipped")
        continue
    chans, kind, _ = dataset_channels(ds)
    a = subset_anatomy(anatomy_for(chans, kind), chans)
    a["cache_key"] = anat_key(chans, kind)
    ANAT[ds] = a
    dens = sorted((WORK / "cache").glob(f"density_{ds}_full_{a['cache_key']}_*s.csv"), key=lambda f: f.stat().st_mtime)
    if dens:                                   # the exact duration the main notebook simulated
        DENSITY_FILE[ds] = dens[-1]
        DUR[ds] = float(re.search(r"_(\d+)s\.csv$", dens[-1].name).group(1))
    else:
        pr = per_rec(ds)
        DUR[ds] = float(np.median(pr.seconds)) if pr is not None and "seconds" in pr else 120.0
    print(f"  {LABEL[ds]}: {len(chans)} channels, simulated duration {DUR[ds]:.0f} s")

Loaded pd_beta_three_datasets.ipynb: datasets ['sandiego', 'newmexico', 'iowa'], head model fsaverage, outputs → pd_beta_three\revision
  San Diego: 32 channels, simulated duration 183 s
  New Mexico: 60 channels, simulated duration 58 s
  Iowa: 58 channels, simulated duration 132 s


## R0 — Software versions and source-space overlap

The inverse space (ico4) is a subset of the generating space (ico5). The check counts how many inverse vertices
are also generating vertices and how close the nearest inverse vertex is to every patch centroid, so the
Methods sentence on the "inverse crime" can state the exact situation.

In [3]:
if RUN["R0"]:
    vers = {"python": sys.version.split()[0], "platform": platform.platform(), "processor": platform.processor()}
    for mod in ("numpy", "scipy", "pandas", "matplotlib", "mne", "mne_icalabel", "picard", "eeglabio", "onnxruntime",
                "torch", "nilearn", "sklearn", "nbformat"):
        try:
            vers[mod] = __import__(mod).__version__
        except Exception:
            vers[mod] = "not installed"
    vers["head_model"] = HEAD_MODEL
    try:
        vers["fsaverage_dir"] = str(mne.datasets.fetch_fsaverage(verbose=False)) if HEAD_MODEL == "fsaverage" else "offline"
    except Exception as e:
        vers["fsaverage_dir"] = f"unavailable ({e})"
    save(pd.DataFrame([vers]).T.reset_index().rename(columns={"index": "item", 0: "version"}), "R0_versions")
    print("\n".join(f"  {k:14s} {v}" for k, v in vers.items()))

    ov = []
    for ds, a in ANAT.items():
        shared = np.isin(a["kept_inv"], a["kept_sim"])
        row = {"dataset": LABEL[ds], "n_generating": len(a["kept_sim"]), "n_inverse": len(a["kept_inv"]),
               "inverse_also_generating": int(shared.sum()), "fraction": shared.mean()}
        for reg in ("L_M1_hand", "L_PMd", "L_SMA"):
            R = a["regions"][reg]
            ps = a["pos_sim"][R["patch_sim"]]
            n_inv_in_patch = int(np.isin(a["kept_inv"], a["kept_sim"][R["patch_sim"]]).sum())
            row[f"{reg}_patch_vertices"] = len(R["patch_sim"])
            row[f"{reg}_inverse_vertices_in_patch"] = n_inv_in_patch
            row[f"{reg}_nearest_inverse_to_centroid_mm"] = float(np.min(np.linalg.norm(a["pos_inv"] - R["centroid"], axis=1)))
        ov.append(row)
    print("\nSource-space overlap (generating ico5 vs inverse ico4):")
    show(save(pd.DataFrame(ov), "R0_source_overlap").T.reset_index(), nd=2)

  python         3.10.11
  platform       Windows-10-10.0.19045-SP0
  processor      Intel64 Family 6 Model 158 Stepping 10, GenuineIntel
  numpy          2.2.6
  scipy          1.15.3
  pandas         2.3.3
  matplotlib     3.10.9
  mne            1.12.1
  mne_icalabel   0.10.0
  picard         0.8.2
  eeglabio       0.1.3
  onnxruntime    1.22.0
  torch          2.14.1+cu126
  nilearn        0.14.1
  sklearn        1.7.2
  nbformat       5.11.1
  head_model     fsaverage
  fsaverage_dir  C:\Users\RTX2080Ti\mne_data\MNE-fsaverage-data\fsaverage

Source-space overlap (generating ico5 vs inverse ico4):
                                   index         0          1         2
                                 dataset San Diego New Mexico      Iowa
                            n_generating     20484      20484     20484
                               n_inverse      5124       5124      5124
                 inverse_also_generating      5124       5124      5124
                               

## R4 — Real data: quality-gate sensitivity, included vs excluded, amplitude decomposition

The GED components of every subject are computed once; each gate setting only changes which component is chosen
and which subjects are included. For every setting, the common threshold is recomputed over the included
recordings and the burst metrics, PD vs HC, OFF vs ON and the meta-analyses are recomputed (no localization).

**Amplitude decomposition.** The primary burst amplitude is the envelope divided by each recording's 1/f beta-band
amplitude. To see whether a group difference sits in the numerator or the denominator, the analysis also reports,
at the subject's pattern-peak electrode and at C3/C4: the raw beta-burst envelope in µV during the detected bursts,
and the aperiodic (1/f) beta-band amplitude in µV.

In [4]:
def candidates(Xs, valids, chans):
    W, A = ged_components(Xs, valids, SFREQ)
    return [{**evaluate(W[:, i], A[:, i], Xs, valids, SFREQ, chans), "label": f"GED {i + 1}", "i": i} for i in range(W.shape[1])]


def passes(c, g):
    if g is None:
        return True
    return (c["beta_db"] >= g["min_beta_db"] and c["slope"] <= g["max_slope"] and c["sm_frac"] >= g["min_sm_frac"]
            and c["peak_ch"] not in EDGE_CHANNELS)


def select(cands, g):
    ok = [c for c in cands if passes(c, g)]
    return (max(ok, key=lambda c: c["score"]), True) if ok else (max(cands, key=lambda c: c["score"]), False)


def load_real(ds):
    chans, kind, files = dataset_channels(ds)
    recs = {}
    for _, r in files.iterrows():
        X, v = load_pre(r.file, chans, kind)
        recs[(r.subject, r.condition)] = {"X": X, "valid": v, "group": r.group, "stem": r.stem}
    for s in {k[0] for k in recs if k[1] == "off"}:
        if (s, "on") in recs:
            n = min(recs[(s, "off")]["X"].shape[1], recs[(s, "on")]["X"].shape[1])
            for c in ("off", "on"):
                recs[(s, c)]["X"], recs[(s, c)]["valid"] = recs[(s, c)]["X"][:, :n], recs[(s, c)]["valid"][:n]
    short = {k for k, d in recs.items() if d["valid"].sum() / SFREQ < MIN_VALID_S}
    subjects = sorted({k[0] for k in recs}, key=lambda s: (s[:2], int(re.sub(r"\D", "", s))))
    cands, conds = {}, {}
    for s in subjects:
        cs = [c for c in ("off", "on", "hc") if (s, c) in recs and (s, c) not in short]
        if cs:
            conds[s] = cs
            cands[s] = candidates([recs[(s, c)]["X"] for c in cs], [recs[(s, c)]["valid"] for c in cs], chans)
    return {"recs": recs, "cands": cands, "conds": conds, "chans": chans, "kind": kind, "short": short}


def gate_run(D, ds, g, sb_cache):
    chosen = {s: select(c, g) for s, c in D["cands"].items()}
    inc = [s for s, (c, ok) in chosen.items() if ok]

    def sb_of(s, c):
        key = (s, c, chosen[s][0]["i"])
        if key not in sb_cache:
            d = D["recs"][(s, c)]
            sb_cache[key] = normalised_beta(d["X"], d["valid"], chosen[s][0]["w"], SFREQ)
        return sb_cache[key]
    if not inc:
        return pd.DataFrame(), np.nan
    pool = np.concatenate([np.abs(signal.hilbert(sb_of(s, c)[0]))[D["recs"][(s, c)]["valid"]] for s in inc for c in D["conds"][s]])
    thr = float(np.percentile(pool, THRESH_PERCENTILE))
    rows = []
    for s in inc:
        for c in D["conds"][s]:
            d = D["recs"][(s, c)]
            sb, sp = sb_of(s, c)
            ev, env = detect_real(sb, d["valid"], SFREQ, thr)
            rows.append({"dataset": ds, "subject": s, "stem": d["stem"], "group": d["group"], "condition": c,
                         "component": chosen[s][0]["label"], **burst_stats(ev, env, d["valid"], SFREQ)})
    return pd.DataFrame(rows), thr


def amplitude_decomposition(D, ds, g):
    rows = []
    chans = D["chans"]
    cc = [chans.index(c) for c in ("C3", "C4") if c in chans]
    chosen = {s: select(c, g) for s, c in D["cands"].items()}
    inc = [s for s, (c, ok) in chosen.items() if ok]
    sbs = {(s, c): normalised_beta(D["recs"][(s, c)]["X"], D["recs"][(s, c)]["valid"], chosen[s][0]["w"], SFREQ)
           for s in inc for c in D["conds"][s]}
    pool = np.concatenate([np.abs(signal.hilbert(sbs[k][0]))[D["recs"][k]["valid"]] for k in sbs])
    thr = float(np.percentile(pool, THRESH_PERCENTILE))
    for (s, c), (sb, sp) in sbs.items():
        d = D["recs"][(s, c)]
        X, v = d["X"], d["valid"]
        ev, env = detect_real(sb, v, SFREQ, thr)
        inb = np.zeros(len(v), bool)
        for e in ev:
            inb[e["onset"]:e["offset"]] = True
        inb &= v
        pk = chans.index(chosen[s][0]["peak_ch"])
        row = {"dataset": ds, "subject": s, "stem": d["stem"], "group": d["group"], "condition": c,
               "burst_amp_norm": float(np.mean(env[inb])) if inb.any() else np.nan,
               "aperiodic_amp_component": float(np.sqrt(sp["ap_var"]))}
        for name, idx in (("peak", [pk]), ("C3C4", cc)):
            xb = sim.bandpass(X[idx], CFG["beta_band"], SFREQ)
            e_uv = np.abs(signal.hilbert(xb, axis=1)) * 1e6
            row[f"burst_env_uV_{name}"] = float(np.mean(e_uv[:, inb])) if inb.any() else np.nan
            row[f"beta_env_uV_{name}"] = float(np.mean(e_uv[:, v]))
            sps = [spectral(X[i] * 1e6, v, SFREQ) for i in idx]
            row[f"aperiodic_uV_{name}"] = float(np.mean([np.sqrt(q["ap_var"]) for q in sps]))
            row[f"beta_peak_db_{name}"] = float(np.mean([q["beta_db"] for q in sps]))
        rows.append(row)
    return pd.DataFrame(rows), thr


def gate_label(g):
    return "no gate" if g is None else f"β≥{g['min_beta_db']:.0f} dB, SM≥{g['min_sm_frac']:.1f}, slope≤{g['max_slope']}"


R4_BURST = ["rate_hz", "dur_ms", "time_in_burst", "env_cv", "burst_amp"]
AMP = ["burst_amp_norm", "aperiodic_amp_component", "burst_env_uV_peak", "beta_env_uV_peak", "aperiodic_uV_peak",
       "beta_peak_db_peak", "burst_env_uV_C3C4", "beta_env_uV_C3C4", "aperiodic_uV_C3C4", "beta_peak_db_C3C4"]

def r4_dataset(ds, D):
    """Gate sensitivity and amplitude decomposition of one dataset → cached CSVs; returns the default threshold."""
    sb_cache, rows, per_gate, thr0 = {}, [], {}, np.nan
    for g in [None, GATE] + [x for x in GATE_GRID if x != GATE]:
        res, thr = gate_run(D, ds, g, sb_cache)
        per_gate[gate_label(g)] = res
        if g == GATE:
            thr0 = thr
        if not len(res):
            continue
        n_inc = res.drop_duplicates("subject").groupby("group").size().to_dict()
        gr = groups(res, R4_BURST, ds)
        pa = paired(res, R4_BURST, ds) if (res.condition == "off").any() else pd.DataFrame()
        for t, kind_ in ((gr, "PD vs HC"), (pa, "OFF vs ON")):
            if len(t):
                rows.append(t.assign(gate=gate_label(g), analysis=kind_, n_PD_subj=n_inc.get("PD", 0),
                                     n_HC_subj=n_inc.get("HC", 0), threshold=thr))
    pd.concat(rows, ignore_index=True).to_csv(REV / f"R4_gate_{ds}.csv", index=False)
    amplitude_decomposition(D, ds, GATE)[0].to_csv(REV / f"R4_amplitude_{ds}.csv", index=False)
    ref = per_rec(ds)
    if ref is not None:
        chk = per_gate[gate_label(GATE)].merge(ref[["stem", "burst_amp"]], on="stem", suffixes=("", "_main"))
        print(f"  {LABEL[ds]}: reproduces the main analysis? burst_amp r = "
              f"{np.corrcoef(chk.burst_amp, chk.burst_amp_main)[0, 1]:.4f}, recordings {len(chk)}/{len(ref)}")
    return thr0

## R5 — Extended split-half reliability

Each analysed recording is split at its median valid sample. In each half the burst metrics and the dSPM and LCMV
maps are computed (common dataset threshold), (a) with the GED filter refitted in each half and gated (as in the
manuscript) and (b) with the subject's whole-data filter kept fixed (isolates detection + localization noise).
Agreement across recordings: ICC(2,1) and Spearman ρ for burst metrics; per recording: peak distance, map
correlation (Pearson over sources) and same-region agreement.

In [5]:
def icc21(a, b):
    Y = np.column_stack([a, b]).astype(float)
    Y = Y[np.all(np.isfinite(Y), axis=1)]
    n, k = Y.shape
    if n < 5:
        return np.nan
    gm = Y.mean()
    msr = k * np.sum((Y.mean(1) - gm) ** 2) / (n - 1)
    msc = n * np.sum((Y.mean(0) - gm) ** 2) / (k - 1)
    mse = np.sum((Y - Y.mean(1, keepdims=True) - Y.mean(0, keepdims=True) + gm) ** 2) / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)


def halves(v):
    idx = np.where(v)[0]
    mid = idx[len(idx) // 2]
    t = np.arange(len(v))
    return v & (t < mid), v & (t >= mid)


def r5_dataset(ds, D, thr):
    a = ANAT[ds]
    rows = []
    for s, cands in D["cands"].items():
        best, ok = select(cands, GATE)
        if not ok:
            continue
        for c in D["conds"][s]:
            d = D["recs"][(s, c)]
            X, v = d["X"], d["valid"]
            for variant in ("refit", "fixed"):
                res = []
                for vh in halves(v):
                    if variant == "refit":
                        det = choose_detector([X], [vh], SFREQ, a["ch_names"])
                        if not det["passed"]:
                            res = None
                            break
                        w = det["w"]
                    else:
                        w = best["w"]
                    sb, _ = normalised_beta(X, vh, w, SFREQ)
                    ev, env = detect_real(sb, vh, SFREQ, thr)
                    mp, _ = localize_real(prepare(X, vh, SFREQ), ev, a["G_inv"], ["dSPM", "LCMV"])
                    res.append({**burst_stats(ev, env, vh, SFREQ), "maps": mp})
                row = {"dataset": ds, "subject": s, "condition": c, "variant": variant, "both_passed": res is not None}
                if res is not None:
                    for k_ in R4_BURST:
                        row[f"{k_}_A"], row[f"{k_}_B"] = res[0][k_], res[1][k_]
                    for m in ("dSPM", "LCMV"):
                        mA, mB = res[0]["maps"][m], res[1]["maps"][m]
                        if np.all(np.isfinite(mA)) and np.all(np.isfinite(mB)):
                            jA, jB = int(np.argmax(mA)), int(np.argmax(mB))
                            row[f"{m}_dist_mm"] = float(np.linalg.norm(a["pos_inv"][jA] - a["pos_inv"][jB]))
                            row[f"{m}_map_r"] = float(np.corrcoef(mA, mB)[0, 1])
                            row[f"{m}_same_region"] = float(map_summary(mA, a, m)["region"] == map_summary(mB, a, m)["region"])
                rows.append(row)
    return pd.DataFrame(rows)


def r6_dataset(ds, D, thr, rng):
    """Surrogates: N_SURR_ENV for envelope CV and duration, N_SURR_MAPS for the dSPM and LCMV maxima."""
    a = ANAT[ds]
    rows = []
    for s, cands in D["cands"].items():
        best, ok = select(cands, GATE)
        if not ok:
            continue
        for c in D["conds"][s]:
            d = D["recs"][(s, c)]
            X, v = d["X"], d["valid"]
            sb, _ = normalised_beta(X, v, best["w"], SFREQ)
            ev, env = detect_real(sb, v, SFREQ, thr)
            st = burst_stats(ev, env, v, SFREQ)
            pre = prepare(X, v, SFREQ)
            nm = noise_model(pre, ev)
            real_maps, _ = localize_real(pre, ev, a["G_inv"], ["dSPM", "LCMV"], noise=nm) if nm is not None else ({}, 0)
            n = pre["n"]
            F_s = np.fft.rfft(sb * v)
            F_e, F_b = np.fft.rfft(pre["Xe"] * v, axis=1), np.fft.rfft(pre["Xb"] * v, axis=1)
            null = {"env_cv": [], "dur_ms": [], "dSPM": [], "LCMV": []}
            for k in range(max(N_SURR_ENV, N_SURR_MAPS)):
                ph = np.exp(2j * np.pi * rng.uniform(size=F_s.size)); ph[0] = 1.0
                if n % 2 == 0:
                    ph[-1] = 1.0
                sbs = np.fft.irfft(F_s * ph, n)
                evs, envs = detect_real(sbs, v, SFREQ, thr)
                if k < N_SURR_ENV:
                    ss = burst_stats(evs, envs, v, SFREQ)
                    null["env_cv"].append(ss["env_cv"]); null["dur_ms"].append(ss["dur_ms"])
                if k < N_SURR_MAPS and nm is not None:
                    ps = {**pre, "Xe": np.fft.irfft(F_e * ph, n, axis=1), "Xb": np.fft.irfft(F_b * ph, n, axis=1)}
                    mp, _ = localize_real(ps, evs, a["G_inv"], ["dSPM", "LCMV"], noise=nm)
                    null["dSPM"].append(np.nanmax(mp["dSPM"])); null["LCMV"].append(np.nanmax(mp["LCMV"]))
            row = {"dataset": ds, "subject": s, "condition": c, "group": d["group"]}
            for k_, r_ in (("env_cv", st["env_cv"]), ("dur_ms", st["dur_ms"]),
                           ("dSPM", np.nanmax(real_maps["dSPM"]) if "dSPM" in real_maps else np.nan),
                           ("LCMV", np.nanmax(real_maps["LCMV"]) if "LCMV" in real_maps else np.nan)):
                nv = np.asarray(null[k_], float); nv = nv[np.isfinite(nv)]
                row[f"p_{k_}"] = (1 + np.sum(nv >= r_)) / (1 + len(nv)) if len(nv) and np.isfinite(r_) else np.nan
                row[f"ratio_{k_}"] = r_ / np.median(nv) if len(nv) and np.isfinite(r_) else np.nan
                row[f"n_surr_{k_}"] = len(nv)
            rows.append(row)
        print(f"\r  {LABEL[ds]} surrogates: {len(rows)} recordings", end="")
    print()
    return pd.DataFrame(rows)


def bh(p):
    p = np.asarray(p, float)
    out = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.sum():
        q = p[ok]
        o = np.argsort(q)
        r = q[o] * ok.sum() / (np.arange(ok.sum()) + 1)
        r = np.minimum.accumulate(r[::-1])[::-1]
        tmp = np.empty_like(q); tmp[o] = np.minimum(r, 1)
        out[ok] = tmp
    return out

### Run R4–R6 (the EEG of one dataset at a time is loaded once and freed afterwards)

In [6]:
RESULTS_THR = {}
for ds in ANAT:
    fthr = REV / f"R4_threshold_{ds}.json"
    files = {"R4": [REV / f"R4_gate_{ds}.csv", REV / f"R4_amplitude_{ds}.csv", fthr],
             "R5": [REV / f"R5_split_half_{ds}.csv"], "R6": [REV / f"R6_surrogates_{ds}.csv"]}
    todo = [k for k in ("R4", "R5", "R6") if RUN[k] and not all(f.exists() for f in files[k])]
    if not todo and fthr.exists():
        RESULTS_THR[ds] = json.loads(fthr.read_text())["thr"]
        print(f"{LABEL[ds]}: cached")
        continue
    t0 = time.time()
    print(f"{LABEL[ds]}: loading EEG and GED components …")
    D = load_real(ds)
    if "R4" in todo or not fthr.exists():
        thr = r4_dataset(ds, D) if "R4" in todo else gate_run(D, ds, GATE, {})[1]
        fthr.write_text(json.dumps({"thr": thr}))
        print(f"  R4 done, threshold {thr:.3f} × 1/f ({time.time() - t0:.0f} s)")
    RESULTS_THR[ds] = json.loads(fthr.read_text())["thr"]
    if "R5" in todo:
        r5_dataset(ds, D, RESULTS_THR[ds]).to_csv(files["R5"][0], index=False)
        print(f"  R5 done ({time.time() - t0:.0f} s)")
    if "R6" in todo:
        r6_dataset(ds, D, RESULTS_THR[ds], np.random.default_rng(SEED + 100)).to_csv(files["R6"][0], index=False)
        print(f"  R6 done ({time.time() - t0:.0f} s)")
    del D
    gc.collect()

San Diego: loading EEG and GED components …
  San Diego: reproduces the main analysis? burst_amp r = 1.0000, recordings 37/37
  R4 done, threshold 2.677 × 1/f (29 s)
  R5 done (98 s)
  San Diego surrogates: 37 recordings
  R6 done (740 s)
New Mexico: loading EEG and GED components …
  New Mexico: reproduces the main analysis? burst_amp r = 1.0000, recordings 58/58
  R4 done, threshold 2.951 × 1/f (41 s)
  R5 done (168 s)
  New Mexico surrogates: 58 recordings
  R6 done (1911 s)
Iowa: loading EEG and GED components …
  Iowa: reproduces the main analysis? burst_amp r = 1.0000, recordings 80/80
  R4 done, threshold 2.843 × 1/f (424 s)
  R5 done (1057 s)
  Iowa surrogates: 80 recordings
  R6 done (9291 s)


In [7]:
"""R4 summary: gate sensitivity and amplitude decomposition."""
if RUN["R4"]:
    G4 = save(pd.concat([pd.read_csv(f) for f in sorted(REV.glob("R4_gate_*.csv")) if "sensitivity" not in f.name],
                        ignore_index=True), "R4_gate_sensitivity")
    metas = []
    for (gl, an), t in G4.groupby(["gate", "analysis"]):
        eff = "g" if an == "PD vs HC" else "d_z"
        tt = t[t.contrast == "PD-ON vs HC"] if an == "PD vs HC" else t
        if tt.dataset.nunique() > 1:
            metas.append(meta(tt, eff).assign(gate=gl, analysis=an))
    M4 = save(pd.concat(metas, ignore_index=True), "R4_gate_sensitivity_meta") if metas else pd.DataFrame()
    print("\nPD-ON vs HC, burst amplitude, for every gate setting (Hedges g, Mann–Whitney p, Holm within setting):")
    x = G4[(G4.analysis == "PD vs HC") & (G4.contrast == "PD-ON vs HC") & (G4.metric == "burst_amp")]
    show(x, ["gate", "dataset", "n_PD", "n_HC", "g", "p", "p_holm"], nd=3)
    print("\nAll burst metrics, Iowa PD-ON vs HC (g) by gate:")
    io_ = G4[(G4.dataset == "iowa") & (G4.contrast == "PD-ON vs HC")]
    if len(io_):
        print(io_.pivot_table(index="gate", columns="metric", values="g").round(2).to_string())
    print("\nOFF vs ON (d_z, Holm p) by gate:")
    oo = G4[G4.analysis == "OFF vs ON"]
    if len(oo):
        oo = oo.assign(v=oo.d_z.round(2).astype(str) + " (" + oo.p_holm.round(2).astype(str) + ")")
        print(oo.drop_duplicates(["gate", "dataset", "metric"]).set_index(["gate", "dataset", "metric"]).v
              .unstack("metric").to_string())
    if len(M4):
        print("\nRandom-effects meta-analysis by gate (PD-ON vs HC: g; OFF vs ON: d_z), burst amplitude and rate:")
        show(M4[M4.metric.isin(["burst_amp", "rate_hz"])],
             ["gate", "analysis", "metric", "k", "RE_pooled", "RE_HKSJ_low", "RE_HKSJ_high", "RE_HKSJ_p", "tau2", "I2"], nd=3)

    A4 = save(pd.concat([pd.read_csv(f) for f in sorted(REV.glob("R4_amplitude_*.csv")) if f.stem.split("_")[-1] in ANAT], ignore_index=True), "R4_amplitude_per_recording")
    amp_stats = []
    for ds, d in A4.groupby("dataset"):
        t = groups(d, AMP, ds)
        if len(t):
            amp_stats.append(t)
    if amp_stats:
        T = save(pd.concat(amp_stats, ignore_index=True), "R4_amplitude_decomposition")
        print("\nAmplitude decomposition, PD vs HC (Hedges g; Holm over the ten variables within dataset × contrast):")
        show(T, ["dataset", "contrast", "metric", "n_PD", "n_HC", "PD", "HC", "g", "p", "p_holm"], nd=3)
        if "iowa" in A4.dataset.values:
            io = with_covariates(A4[A4.dataset == "iowa"], "iowa")
            covs = [c for c in ("age", "male", "moca") if c in io and io[c].notna().any()]
            o = []
            for m in AMP:
                b, se, p, n = ols_group(io, m, covs)
                o.append({"metric": m, "beta_PD_minus_HC": b, "se": se, "p": p, "n": n,
                          "beta_std": b / io[m].std() if np.isfinite(b) else np.nan})
            o = pd.DataFrame(o); o["p_holm"] = holm(o.p)
            print(f"\nIowa, adjusted for {', '.join(covs)} (beta_std = β / SD of the variable):")
            show(save(o, "R4_amplitude_iowa_ols"), nd=4)


PD-ON vs HC, burst amplitude, for every gate setting (Hedges g, Mann–Whitney p, Holm within setting):
                      gate   dataset  n_PD  n_HC      g     p  p_holm
                   no gate      iowa  87.0  44.0 -0.848 0.000   0.000
β≥2 dB, SM≥0.4, slope≤-0.5      iowa  48.0  32.0 -0.813 0.001   0.006
β≥1 dB, SM≥0.3, slope≤-0.5      iowa  52.0  38.0 -0.707 0.003   0.017
β≥1 dB, SM≥0.4, slope≤-0.5      iowa  49.0  33.0 -0.741 0.002   0.010
β≥1 dB, SM≥0.5, slope≤-0.5      iowa  40.0  27.0 -0.572 0.027   0.137
β≥2 dB, SM≥0.3, slope≤-0.5      iowa  52.0  37.0 -0.790 0.001   0.006
β≥2 dB, SM≥0.5, slope≤-0.5      iowa  40.0  26.0 -0.648 0.014   0.069
β≥3 dB, SM≥0.3, slope≤-0.5      iowa  49.0  36.0 -0.654 0.007   0.034
β≥3 dB, SM≥0.4, slope≤-0.5      iowa  45.0  31.0 -0.827 0.001   0.007
β≥3 dB, SM≥0.5, slope≤-0.5      iowa  36.0  25.0 -0.668 0.023   0.113
β≥2 dB, SM≥0.4, slope≤-0.3      iowa  51.0  32.0 -0.865 0.000   0.002
β≥2 dB, SM≥0.4, slope≤-0.7      iowa  40.0  29.0 -0.876 0

In [8]:
"""Included vs excluded participants at the pre-registered gate (from the saved tables; no EEG needed)."""
if RUN["R4"]:
    rows = []
    for ds in ANAT:
        qc = per_rec(ds, kind="detector_qc")
        ng = per_rec(ds, kind="per_recording_no_gate")
        cov_f = WORK / "preprocessed" / ds / "covariates.csv"
        if qc is None or not cov_f.exists():
            continue
        cov = pd.read_csv(cov_f)
        m = cov.out_name.astype(str).str.extract(r"(pd|control)(\d+)")
        cov["subj"] = m[0].str.replace("control", "hc") + m[1]
        cov = cov.drop_duplicates("subj")
        q = qc[qc.get("group").notna()] if "group" in qc else qc
        q = q.merge(cov[["subj"] + [c for c in ("age", "sex", "moca", "updrs") if c in cov]], left_on="subject",
                    right_on="subj", how="left")
        if ng is not None:
            nb_ = ng.groupby("subject")[["burst_amp", "rate_hz", "env_cv", "snr_est_db"]].mean().reset_index()
            q = q.merge(nb_, on="subject", how="left")
        for grp_, d in q.groupby("group"):
            inc_, exc_ = d[d.included.astype(bool)], d[~d.included.astype(bool)]
            for var in ("age", "moca", "updrs", "sm_frac", "beta_db_min", "slope_max", "burst_amp", "rate_hz",
                        "env_cv", "snr_est_db"):
                if var not in d or (var == "updrs" and grp_ == "HC"):
                    continue
                a_, b_ = pd.to_numeric(inc_[var], errors="coerce").dropna(), pd.to_numeric(exc_[var], errors="coerce").dropna()
                if len(a_) >= 3 and len(b_) >= 3:
                    rows.append({"dataset": LABEL[ds], "group": grp_, "variable": var, "n_included": len(a_),
                                 "n_excluded": len(b_), "included_median": a_.median(), "excluded_median": b_.median(),
                                 "mann_whitney_p": stats.mannwhitneyu(a_, b_).pvalue})
            if "sex" in d:
                sx = lambda z: f"{(z.astype(str).str.upper().str[0] == 'F').sum()}/{len(z)}"
                rows.append({"dataset": LABEL[ds], "group": grp_, "variable": "female/n",
                             "n_included": len(inc_), "n_excluded": len(exc_), "included_median": sx(inc_.sex),
                             "excluded_median": sx(exc_.sex), "mann_whitney_p": np.nan})
    IE = pd.DataFrame(rows)
    if len(IE):
        IE["p_holm_within_dataset_group"] = IE.groupby(["dataset", "group"]).mann_whitney_p.transform(lambda p: holm(p.values))
    print("Included vs excluded participants (burst metrics of the excluded: best component without the gate):")
    show(save(IE, "R4_included_vs_excluded"), nd=3)

Included vs excluded participants (burst metrics of the excluded: best component without the gate):
   dataset group    variable  n_included  n_excluded included_median excluded_median  mann_whitney_p  p_holm_within_dataset_group
 San Diego    HC    female/n           9           2             3/9             1/2             NaN                          NaN
 San Diego    PD    female/n          14           0            7/14             0/0             NaN                          NaN
New Mexico    HC    female/n          16           2            7/16             1/2             NaN                          NaN
New Mexico    PD    female/n          22           2            8/22             1/2             NaN                          NaN
      Iowa    HC         age          32          12            71.0            73.5           0.428                        0.856
      Iowa    HC        moca          32          12            27.0            27.0           0.799                    

In [9]:
"""R5 and R6 summaries."""
R5L = [pd.read_csv(f) for ds in ANAT for f in [REV / f"R5_split_half_{ds}.csv"] if RUN["R5"] and f.exists()]
R6L = [pd.read_csv(f) for ds in ANAT for f in [REV / f"R6_surrogates_{ds}.csv"] if RUN["R6"] and f.exists()]
if R5L:
    R5 = pd.concat(R5L, ignore_index=True)
    out = []
    for (ds, var), d in R5.groupby(["dataset", "variant"]):
        ok = d[d.both_passed.astype(bool)]
        row = {"dataset": LABEL[ds], "variant": var, "recordings": len(d), "both_halves_passed": len(ok)}
        for k_ in R4_BURST:
            if f"{k_}_A" in ok:
                row[f"ICC_{k_}"] = icc21(ok[f"{k_}_A"], ok[f"{k_}_B"])
                row[f"rho_{k_}"] = stats.spearmanr(ok[f"{k_}_A"], ok[f"{k_}_B"], nan_policy="omit")[0]
        for m in ("dSPM", "LCMV"):
            if f"{m}_dist_mm" in ok:
                row[f"{m}_dist_mm_median"] = ok[f"{m}_dist_mm"].median()
                row[f"{m}_map_r_median"] = ok[f"{m}_map_r"].median()
                row[f"{m}_same_region"] = ok[f"{m}_same_region"].mean()
        out.append(row)
    S5 = save(pd.DataFrame(out), "R5_split_half_summary")
    print("Split-half reliability (ICC(2,1) and Spearman ρ across recordings; map agreement per recording):")
    show(S5.T.reset_index(), nd=2)
if R6L:
    R6 = pd.concat(R6L, ignore_index=True)
    out = []
    for ds, d in R6.groupby("dataset"):
        row = {"dataset": LABEL[ds], "recordings": len(d)}
        for k_ in ("env_cv", "dur_ms", "dSPM", "LCMV"):
            p = d[f"p_{k_}"]
            row[f"{k_}_n_surr"] = int(d[f"n_surr_{k_}"].median()) if len(d) else 0
            row[f"{k_}_p<.05"] = f"{(p < 0.05).sum()}/{p.notna().sum()}"
            row[f"{k_}_FDR<.05"] = f"{(bh(p.values) < 0.05).sum()}/{p.notna().sum()}"
            row[f"{k_}_ratio_median"] = d[f"ratio_{k_}"].median()
            k_sig = int((p < 0.05).sum()); n_ = int(p.notna().sum())
            row[f"{k_}_binom_p_vs_5%"] = stats.binomtest(k_sig, n_, 0.05, alternative="greater").pvalue if n_ else np.nan
        out.append(row)
    S6 = save(pd.DataFrame(out), "R6_surrogate_summary")
    print("Surrogate tests (phase-randomised, same filter, threshold, inverse and noise model):")
    show(S6.T.reset_index(), nd=3)

Split-half reliability (ICC(2,1) and Spearman ρ across recordings; map agreement per recording):
              index          0          1          2          3          4          5
            dataset       Iowa       Iowa New Mexico New Mexico  San Diego  San Diego
            variant      fixed      refit      fixed      refit      fixed      refit
         recordings         80         80         58         58         37         37
 both_halves_passed         80         65         58         48         37         32
        ICC_rate_hz   0.942618   0.822002   0.888249   0.826949   0.729114   0.658993
        rho_rate_hz   0.943032   0.793706   0.899729   0.835319   0.686266   0.636972
         ICC_dur_ms   0.915293   0.799324   0.795699   0.608305   0.906534   0.727197
         rho_dur_ms   0.890262   0.688156   0.789324   0.686095   0.756994   0.812683
  ICC_time_in_burst   0.965925   0.883105    0.91107   0.810473    0.86186   0.762725
  rho_time_in_burst   0.954635   0.822159  

## R1 — Simulation with the real-data detector

Every simulated recording (the dataset's own montage, generating head model and median recording length) is
analysed exactly like a real recording: six GED components of 13–30 Hz against 5–11 + 32–45 Hz, the quality gate,
the best passing component, the beta signal divided by its 1/f beta-band amplitude, bursts ≥ 2 cycles, and
trough-locked dSPM/eLORETA and LCMV burst ÷ burst-free power, with the real-data inverse code. Three threshold
scopes are compared:

* **per_recording**: 75th percentile of the recording's own normalised envelope (as in the original simulation);
* **cohort**: one 75th percentile pooled over every gated recording of the simulated cohort (all regions and
  SNRs), i.e. the real-data rule;
* **real**: the threshold actually used for the real dataset (same 1/f-relative units).

Ground truth gives precision/recall/F1 and localization scores; burst-free recordings give the false-burst rate.

In [10]:
def sim_record(a, region, snr, seed, cfg, with_bursts=True, gain_override=None, extra=None):
    X, s, ev = sim.simulate(a, region, "OFF", float(snr), int(seed), cfg, with_bursts, gain_override)
    if extra is not None:
        X = extra(X, np.random.default_rng(int(seed) + 7))
    v = np.ones(X.shape[1], bool)
    pad = int(EDGE_PAD_S * SFREQ)
    v[:pad] = v[-pad:] = False
    return X, ev, v


def real_detector(X, v, chans):
    """The real-data detector: best GED component that passes the pre-registered gate (GATE)."""
    return choose_detector([X], [v], SFREQ, chans, gate=True)


def score_run(a, X, v, ev_true, region, sb, thr, cfg, methods=R1_METHODS, oracle=True):
    ev, env = detect_real(sb, v, SFREQ, thr)
    out = {"n_detected": len(ev), "time_in_burst": burst_stats(ev, env, v, SFREQ)["time_in_burst"]}
    if ev_true:
        out.update(sim.detection_scores(ev_true, ev, SFREQ, cfg["match_tolerance_s"]))
    pre = prepare(X, v, SFREQ)
    maps, nave = localize_real(pre, ev, a["G_inv"], methods)
    rows = []
    for m, mp in maps.items():
        sc = sim.score_map(mp, a, region, cfg, m) if np.all(np.isfinite(mp)) else {}
        rows.append({**out, "method": m, "nave": nave, **{k: sc.get(k, np.nan) for k in
                                                           ("peak_err_mm", "region_correct", "hemi_correct", "dispersion_mm")}})
    if oracle and ev_true:
        mp, nave = localize_real(pre, ev_true, a["G_inv"], ["dSPM"])
        sc = sim.score_map(mp["dSPM"], a, region, cfg, "dSPM") if np.all(np.isfinite(mp["dSPM"])) else {}
        rows.append({**out, "method": "dSPM_oracle", "nave": nave,
                     **{k: sc.get(k, np.nan) for k in ("peak_err_mm", "region_correct", "hemi_correct", "dispersion_mm")}})
    return rows


def r1_dataset(ds):
    f = REV / f"R1_sim_real_detector_{ds}.csv"
    if f.exists():
        return pd.read_csv(f)
    a, cfg = ANAT[ds], {**CFG, "duration": DUR[ds]}
    real_thr = RESULTS_THR.get(ds, np.nan)
    jobs = [(reg, snr, 700_000 + 1000 * ri + 10 * si + k, True) for ri, reg in enumerate(R1_REGIONS)
            for si, snr in enumerate(R1_SNRS) for k in range(R1_SEEDS)]
    jobs += [("L_M1_hand", 0.0, 790_000 + k, False) for k in range(R1_NOISE_ONLY)]
    t0, first, env_pool = time.time(), {}, []
    for i, (reg, snr, seed, wb) in enumerate(jobs):                    # pass 1: detector + gate
        X, ev, v = sim_record(a, reg, snr, seed, cfg, wb)
        det = real_detector(X, v, a["ch_names"])
        sb, sp = normalised_beta(X, v, det["w"], SFREQ)
        first[seed] = {"w": det["w"], "passed": det["passed"], "reason": det["reason"], "sm_frac": det["sm_frac"],
                       "beta_db": det["beta_db"], "slope": det["slope"]}
        if det["passed"] and wb:
            e = np.abs(signal.hilbert(sb))[v]
            env_pool.append(np.random.default_rng(seed).choice(e, min(len(e), 20_000), replace=False))
        print(f"\r  {LABEL[ds]} pass 1: {i + 1}/{len(jobs)} ({time.time() - t0:.0f} s)", end="")
    cohort_thr = float(np.percentile(np.concatenate(env_pool), THRESH_PERCENTILE)) if env_pool else np.nan
    print(f"\n  cohort threshold {cohort_thr:.2f}, real-data threshold {real_thr:.2f} (× 1/f amplitude)")
    rows, t0 = [], time.time()
    for i, (reg, snr, seed, wb) in enumerate(jobs):                    # pass 2: detection + localization
        X, ev, v = sim_record(a, reg, snr, seed, cfg, wb)
        d = first[seed]
        sb, _ = normalised_beta(X, v, d["w"], SFREQ)
        env = np.abs(signal.hilbert(sb))
        for scope, thr in (("per_recording", np.percentile(env[v], THRESH_PERCENTILE)), ("cohort", cohort_thr),
                           ("real", real_thr)):
            if not np.isfinite(thr):
                continue
            for r in score_run(a, X, v, ev, reg, sb, thr, cfg):
                rows.append({"dataset": ds, "region": reg, "snr_db": snr, "seed": seed, "bursts": wb, "scope": scope,
                             "threshold": thr, "gate_passed": d["passed"], "gate_reason": d["reason"],
                             "sm_frac": d["sm_frac"], "beta_db": d["beta_db"], "slope": d["slope"], **r})
        print(f"\r  {LABEL[ds]} pass 2: {i + 1}/{len(jobs)} ({time.time() - t0:.0f} s)", end="")
    print()
    df = pd.DataFrame(rows)
    df.to_csv(f, index=False)
    return df


def r1_summary(df):
    d = df[df.bursts & df.gate_passed]
    out = []
    for (ds, scope, m, snr), g in d.groupby(["dataset", "scope", "method", "snr_db"]):
        mot = g[g.region.isin(MOTOR)]
        lo, hi = boot_ci(mot.peak_err_mm)
        out.append({"dataset": LABEL[ds], "scope": scope, "method": m, "snr_db": snr, "n": len(mot),
                    "F1": g.f1.mean(), "precision": g.precision.mean(), "recall": g.recall.mean(),
                    "err_mm_motor": mot.peak_err_mm.median(), "err_ci_low": lo, "err_ci_high": hi,
                    "region_acc_motor": mot.region_correct.mean(),
                    "region_acc_ci": "[{:.2f}, {:.2f}]".format(*boot_ci(mot.region_correct, np.mean))})
    return pd.DataFrame(out)


for ds in ANAT:                       # real-data thresholds, cached by the R4–R6 run cell
    fthr = REV / f"R4_threshold_{ds}.json"
    if ds not in RESULTS_THR and fthr.exists():
        RESULTS_THR[ds] = json.loads(fthr.read_text())["thr"]
missing = [LABEL[ds] for ds in ANAT if ds not in RESULTS_THR]
if missing:
    print(f"No real-data threshold for {missing} (run the R4–R6 cell first): the 'real' scope is skipped there.")

In [11]:
if RUN["R1"]:
    R1 = pd.concat([r1_dataset(ds) for ds in ANAT], ignore_index=True)
    S1 = save(r1_summary(R1), "R1_summary")
    print("Gate pass rate of simulated recordings with bursts, by region and SNR (all datasets):")
    gp = R1[R1.bursts & (R1.scope == "per_recording") & (R1.method == "dSPM")]
    print(gp.pivot_table(index="region", columns="snr_db", values="gate_passed", aggfunc="mean").round(2).to_string())
    print("\nDetection and localization (gated recordings; error and region accuracy over the five motor regions):")
    show(S1[S1.method.isin(["dSPM", "LCMV", "dSPM_oracle"])],
         ["dataset", "scope", "method", "snr_db", "n", "F1", "err_mm_motor", "err_ci_low", "err_ci_high",
          "region_acc_motor", "region_acc_ci"], nd=2)
    nz = R1[~R1.bursts & (R1.method == "dSPM")]
    fp = nz.assign(false_bursts_per_s=nz.n_detected / nz.dataset.map(DUR)).groupby(["dataset", "scope"]).agg(
        recordings=("seed", "size"), gate_pass=("gate_passed", "mean"), false_bursts_per_s=("false_bursts_per_s", "mean"),
        time_in_burst=("time_in_burst", "mean")).reset_index()
    print("\nBurst-free recordings: gate pass rate and false bursts under each threshold scope:")
    show(save(fp, "R1_noise_only"), nd=3)
    old = [pd.read_csv(DENSITY_FILE[ds]) for ds in ANAT if ds in DENSITY_FILE]
    if old:
        old = pd.concat(old, ignore_index=True)
        oc = old[old.region.isin(MOTOR)].groupby(["dataset", "method", "snr_db"]).agg(
            err_mm_original=("peak_err_mm", "median"), region_acc_original=("region_correct", "mean")).reset_index()
        new = R1[R1.bursts & R1.gate_passed & (R1.scope == "cohort") & R1.region.isin(MOTOR)].groupby(
            ["dataset", "method", "snr_db"]).agg(err_mm_real_detector=("peak_err_mm", "median"),
                                                 region_acc_real_detector=("region_correct", "mean")).reset_index()
        cmp_ = oc.merge(new, on=["dataset", "method", "snr_db"])
        print("\nOriginal density simulation (broadband GED, per-recording threshold) vs real-data detector (cohort threshold):")
        show(save(cmp_, "R1_vs_original"), nd=2)

  San Diego pass 1: 440/440 (1430 s)
  cohort threshold 2.04, real-data threshold 2.68 (× 1/f amplitude)
  San Diego pass 2: 440/440 (1927 s)
  New Mexico pass 1: 440/440 (503 s)
  cohort threshold 2.07, real-data threshold 2.95 (× 1/f amplitude)
  New Mexico pass 2: 440/440 (1229 s)
  Iowa pass 1: 440/440 (1041 s)
  cohort threshold 2.04, real-data threshold 2.84 (× 1/f amplitude)
  Iowa pass 2: 440/440 (2592 s)
Gate pass rate of simulated recordings with bursts, by region and SNR (all datasets):
snr_db       -15.0  -10.0  -5.0    0.0    5.0    10.0
region                                               
L_M1_hand     0.70   0.87   0.87   1.00   0.87   0.73
L_PMd         0.80   0.73   0.73   1.00   1.00   0.90
L_SMA         0.80   0.97   1.00   1.00   0.97   0.70
L_occipital   0.77   0.83   0.83   0.80   0.67   0.67
L_thalamus    0.83   0.73   0.70   0.77   0.90   0.63
R_M1_hand     0.77   0.77   0.93   1.00   0.87   0.77
R_PMd         0.83   0.73   0.83   1.00   1.00   0.90

Detection 

## R2 — Region-specific SNR calibration with confidence intervals

The calibration curve (median per-recording burst/background ratio at each simulated SNR) is built separately
for M1, PMd and SMA, and for all five motor regions together. Every recording's SNR is read off each curve; the
bootstrap (simulated recordings resampled within SNR levels) gives a 95 % CI of every dataset's median SNR, and
the expected accuracy at that SNR is read from R1 (real-data detector, cohort threshold).

In [12]:
def r2_calibration(ds):
    f = REV / f"R2_calibration_{ds}.csv"
    if f.exists():
        return pd.read_csv(f)
    a, cfg = ANAT[ds], {**CFG, "duration": DUR[ds]}
    rows, t0 = [], time.time()
    jobs = [(reg, snr, r) for reg in R2_REGIONS for snr in CALIB_SNRS for r in range(R2_REPS)]
    for i, (reg, snr, r) in enumerate(jobs):
        X, ev, v = sim_record(a, reg, snr, 950_000 + 1000 * R2_REGIONS.index(reg) + 10 * r + int(snr * 3), cfg)
        det = choose_detector([X], [v], SFREQ, a["ch_names"], gate=False)       # as calibrate_snr()
        rows.append({"dataset": ds, "region": reg, "snr_db": snr, "rep": r, "bbr_db": per_recording_bbr(X, v, det["w"], SFREQ)})
        print(f"\r  {LABEL[ds]} calibration {i + 1}/{len(jobs)} ({time.time() - t0:.0f} s)", end="")
    print()
    df = pd.DataFrame(rows)
    df.to_csv(f, index=False)
    return df


def invert(bbr, cal):
    c = cal.groupby("snr_db").bbr_db.median()
    y = np.maximum.accumulate(c.values)
    return np.interp(bbr, y, c.index.values, left=c.index[0], right=c.index[-1])


def expected_from_r1(snr, ds, method, R1):
    d = R1[(R1.dataset == ds) & R1.bursts & R1.gate_passed & (R1.scope == "cohort") & (R1.method == method)
           & R1.region.isin(MOTOR)]
    if not len(d) or not np.isfinite(snr):
        return np.nan, np.nan
    g = d.groupby("snr_db")
    x = np.array(sorted(g.groups))
    s = float(np.clip(snr, x.min(), x.max()))
    return (float(np.interp(s, x, g.peak_err_mm.median().reindex(x).values)),
            float(np.interp(s, x, g.region_correct.mean().reindex(x).values)))


if RUN["R2"]:
    if "R1" not in globals():
        R1 = pd.concat([pd.read_csv(f) for f in REV.glob("R1_sim_real_detector_*.csv")], ignore_index=True) \
            if list(REV.glob("R1_sim_real_detector_*.csv")) else pd.DataFrame(columns=["dataset"])
    groups_cal = {"M1": ["L_M1_hand", "R_M1_hand"], "PMd": ["L_PMd", "R_PMd"], "SMA": ["L_SMA"], "all motor": R2_REGIONS}
    rows, curves = [], []
    for ds in ANAT:
        cal = r2_calibration(ds)
        res = per_rec(ds)
        bbr = res.bbr_db_rec.dropna().values
        for name, regs in groups_cal.items():
            c = cal[cal.region.isin(regs)]
            curves.append(c.groupby("snr_db").bbr_db.agg(["median", "std"]).reset_index().assign(dataset=LABEL[ds], calibration=name))
            snr = invert(bbr, c)
            rng = np.random.default_rng(1)
            levels = sorted(c.snr_db.unique())
            vals = [c.loc[c.snr_db == x, "bbr_db"].dropna().values for x in levels]
            boot = []
            for _ in range(N_BOOT // 2):                     # resample simulated recordings within each SNR level
                med = np.maximum.accumulate([np.median(rng.choice(v_, len(v_))) for v_ in vals])
                boot.append(np.median(np.interp(bbr, med, levels, left=levels[0], right=levels[-1])))
            med = float(np.median(snr))
            row = {"dataset": LABEL[ds], "calibration": name, "n_recordings": len(bbr), "median_snr_db": med,
                   "snr_ci_low": np.percentile(boot, 2.5), "snr_ci_high": np.percentile(boot, 97.5),
                   "snr_iqr_low": np.percentile(snr, 25), "snr_iqr_high": np.percentile(snr, 75)}
            if len(R1):
                for m in ("dSPM", "LCMV"):
                    e, acc = expected_from_r1(med, ds, m, R1)
                    e_lo, _ = expected_from_r1(row["snr_ci_high"], ds, m, R1)
                    e_hi, _ = expected_from_r1(row["snr_ci_low"], ds, m, R1)
                    row.update({f"{m}_expected_err_mm": e, f"{m}_err_range": f"[{e_lo:.1f}, {e_hi:.1f}]",
                                f"{m}_expected_region_acc": acc})
            rows.append(row)
    R2 = save(pd.DataFrame(rows), "R2_snr_by_calibration")
    save(pd.concat(curves, ignore_index=True), "R2_calibration_curves")
    print("Median SNR of the analysed recordings under each calibration region (95 % bootstrap CI of the median),\n"
          "and the expected accuracy at that SNR with the real-data detector (R1, cohort threshold):")
    show(R2, nd=2)

  San Diego calibration 480/480 (1525 s)
  New Mexico calibration 480/480 (488 s)
  Iowa calibration 480/480 (1092 s)
Median SNR of the analysed recordings under each calibration region (95 % bootstrap CI of the median),
and the expected accuracy at that SNR with the real-data detector (R1, cohort threshold):
   dataset calibration  n_recordings  median_snr_db  snr_ci_low  snr_ci_high  snr_iqr_low  snr_iqr_high  dSPM_expected_err_mm dSPM_err_range  dSPM_expected_region_acc  LCMV_expected_err_mm LCMV_err_range  LCMV_expected_region_acc
 San Diego          M1            37          -3.42       -3.76        -2.96        -3.79         -3.13                 18.83   [18.0, 19.4]                      0.68                 21.28   [20.6, 21.8]                      0.62
 San Diego         PMd            37          -1.35       -2.58        -1.12        -2.10         -0.73                 15.24   [14.9, 17.4]                      0.83                 18.09   [17.7, 20.0]                      0.68

## R3 — Sensitivity of the simulation to its assumptions

Run with one dataset's montage and recording length (`R3_DATASET`) at the SNRs of the real data, with the real-data
detector and a per-recording threshold (scope does not matter for localization here; R1 compares scopes):

* **patch radius** 5 / 10 / 15 mm (the source extent);
* **two generators**: left and right M1 together, independent bursts or the same (synchronous) burst train. Error =
  distance to the nearer true source; "both" = the map's maximum within 20 mm of the other source is ≥ 50 % of
  the global maximum;
* **deep sources**: thalamus (in the main simulation) and subthalamic nucleus;
* **EMG contamination**: broadband 20–150 Hz noise on temporal and frontal edge electrodes, intermittent (bursts of
  0.5–2 s), at 1, 3 and 10 × the background beta-band RMS (broadband 20–150 Hz RMS). Reports how often the gate rejects the recording and the
  accuracy when it does not;
* **beta frequency**: bursts at 13–17 Hz (low beta), 16–24 Hz (default) and 22–30 Hz (high beta).

In [13]:
def patch(a, seed_mm, radius):
    seed_mm = np.asarray(seed_mm, float)
    h = 0 if seed_mm[0] < 0 else 1
    cand = np.where(a["hemi_sim"] == h)[0]
    centre = cand[np.argmin(np.linalg.norm(a["pos_sim"][cand] - seed_mm, axis=1))]
    p = cand[np.linalg.norm(a["pos_sim"][cand] - a["pos_sim"][centre], axis=1) <= radius]
    return {"kind": "cortical", "seed": seed_mm, "patch_sim": p, "centroid": a["pos_sim"][p].mean(0),
            "gain": a["G_sim"][:, p].mean(axis=1), "hemi": h}


def stn_region(a, chans, kind):
    """Free-orientation gain at the STN from the generating BEM (projected on the deep-source orientation)."""
    import os
    SIMNS["eeg_info"] = make_eeg_info(chans, kind)
    info_sim = SIMNS["eeg_info"](CFG, CFG["sim_electrode_jitter_mm"], CFG["base_seed"])
    bem = mne.read_bem_solution(WORK / "cache" / f"bem_sim_{HEAD_MODEL}-sol.fif")
    trans = os.path.join(os.path.dirname(mne.__file__), "data", "fsaverage", "fsaverage-trans.fif")
    Gf = sim.free_gain(info_sim, trans, np.asarray(STN_POS) / 1000.0, bem)
    ori = np.asarray(CFG["deep_orientation"], float); ori /= np.linalg.norm(ori)
    return {"kind": "deep", "seed": np.asarray(STN_POS), "patch_sim": np.array([], int), "centroid": np.asarray(STN_POS),
            "gain": Gf @ ori, "hemi": 0}


def emg_adder(level, chans, bg_ref):
    edge = [chans.index(c) for c in EDGE_CHANNELS if c in chans]

    def add(X, rng):
        n = X.shape[1]
        on = np.zeros(n)
        t = 0
        while t < n:
            t += int(rng.uniform(1, 6) * SFREQ)
            L = int(rng.uniform(0.5, 2.0) * SFREQ)
            on[t:t + L] = 1.0
        on = np.convolve(on, np.hanning(int(0.1 * SFREQ)) / (0.05 * SFREQ), "same")
        E = sim.bandpass(rng.standard_normal((len(edge), n)), (20.0, min(150.0, SFREQ / 2.2)), SFREQ) * on
        E *= (rng.uniform(0.5, 1.5, len(edge)) / E.std(axis=1).clip(1e-12))[:, None]
        Y = X.copy()
        Y[edge] += level * bg_ref * E
        return Y - Y.mean(axis=0, keepdims=True)
    return add


def r3_run(label, value, a, region, snr, seed, cfg, gain_override=None, extra=None, truths=None):
    X, ev, v = sim_record(a, region, snr, seed, cfg, gain_override=gain_override, extra=extra)
    det = real_detector(X, v, a["ch_names"])
    sb, _ = normalised_beta(X, v, det["w"], SFREQ)
    env = np.abs(signal.hilbert(sb))
    thr = np.percentile(env[v], THRESH_PERCENTILE)
    out = []
    if truths is None:
        for r in score_run(a, X, v, ev, region, sb, thr, cfg, methods=["dSPM", "LCMV"], oracle=False):
            out.append({"factor": label, "level": value, "snr_db": snr, "seed": seed, "gate_passed": det["passed"],
                        "gate_reason": det["reason"], "peak_channel": det["peak_ch"], **r})
        return out
    evd, _ = detect_real(sb, v, SFREQ, thr)
    maps, nave = localize_real(prepare(X, v, SFREQ), evd, a["G_inv"], ["dSPM", "LCMV"])
    for m, mp in maps.items():
        if not np.all(np.isfinite(mp)):
            continue
        j = int(np.argmax(mp))
        d = [np.linalg.norm(a["pos_inv"][j] - a["regions"][t]["centroid"]) for t in truths]
        other = truths[int(np.argmax(d))]
        near = np.linalg.norm(a["pos_inv"] - a["regions"][other]["centroid"], axis=1) < 20
        base = 1.0 if m == "LCMV" else 0.0
        both = (mp[near].max() - base) >= 0.5 * (mp[j] - base)
        out.append({"factor": label, "level": value, "snr_db": snr, "seed": seed, "gate_passed": det["passed"],
                    "method": m, "nave": nave, "peak_err_mm": min(d), "both_sources": float(both),
                    "peak_hemi": "L" if a["hemi_inv"][j] == 0 else "R"})
    return out


if RUN["R3"] and R3_DATASET in ANAT:
    f = REV / "R3_sensitivity.csv"
    if f.exists():
        R3 = pd.read_csv(f)
    else:
        a0 = ANAT[R3_DATASET]
        cfg0 = {**CFG, "duration": DUR[R3_DATASET]}
        chans, kind = a0["ch_names"], DATASETS[R3_DATASET]["montage"]
        rows, t0 = [], time.time()
        seeds = lambda k: [810_000 + 100 * k + s for s in range(R3_SEEDS)]
        for snr in R3_SNRS:
            for radius in (5.0, 10.0, 15.0):                                   # patch radius
                a = {**a0, "regions": {**a0["regions"], "L_M1_hand": patch(a0, CFG["regions"]["L_M1_hand"], radius)}}
                for s in seeds(int(radius)):
                    rows += r3_run("patch_radius_mm", radius, a, "L_M1_hand", snr, s, {**cfg0, "patch_radius_mm": radius})
            gL, gR = a0["regions"]["L_M1_hand"]["gain"], a0["regions"]["R_M1_hand"]["gain"]
            for s in seeds(20):                                                # two synchronous generators
                rows += r3_run("two_generators", "synchronous", a0, "L_M1_hand", snr, s, cfg0, gain_override=gL + gR,
                               truths=["L_M1_hand", "R_M1_hand"])
            def second(X, rng, snr=snr):                                       # independent burst train in R M1
                g = gR - gR.mean()
                s2, _ = sim.make_source(rng, X.shape[1], SFREQ, CFG["conditions"]["OFF"], cfg0)
                best = int(np.argmax(np.abs(g)))
                bg = np.std(sim.bandpass(X[best], CFG["beta_band"], SFREQ))
                inb = s2 != 0
                Y = X + 10 ** (snr / 20) * bg / np.sqrt(np.mean((g[best] * s2[inb]) ** 2)) * np.outer(g, s2)
                return Y - Y.mean(axis=0, keepdims=True)
            for s in seeds(21):
                rows += r3_run("two_generators", "independent", a0, "L_M1_hand", snr, s, cfg0, extra=second,
                               truths=["L_M1_hand", "R_M1_hand"])
            try:
                a_stn = {**a0, "regions": {**a0["regions"], "L_STN": stn_region(a0, chans, kind)}}
                for reg in ("L_thalamus", "L_STN"):
                    for s in seeds(30):
                        rows += r3_run("deep_source", reg, a_stn, reg, snr, s, cfg0)
            except Exception as e:
                print(f"  STN skipped: {type(e).__name__}: {e}")
            Xb0, _, _ = sim.simulate(a0, "L_M1_hand", "OFF", 0.0, 1, cfg0, with_bursts=False)
            bg_beta = float(np.median(np.std(sim.bandpass(Xb0, CFG["beta_band"], SFREQ), axis=1)))
            for lev in (0.0, 1.0, 3.0, 10.0):
                for s in seeds(40):
                    rows += r3_run("emg_x_background_beta", lev, a0, "L_M1_hand", snr, s, cfg0,
                                   extra=emg_adder(lev, chans, bg_beta) if lev > 0 else None)
            for band in ((13.0, 17.0), (16.0, 24.0), (22.0, 30.0)):
                for s in seeds(50):
                    rows += r3_run("burst_freq_hz", f"{band[0]:.0f}-{band[1]:.0f}", a0, "L_M1_hand", snr, s,
                                   {**cfg0, "burst_freq": band})
            print(f"\r  R3 SNR {snr}: {len(rows)} rows ({time.time() - t0:.0f} s)", end="")
        print()
        R3 = pd.DataFrame(rows)
        R3.to_csv(f, index=False)
    keys = ["factor", "level", "snr_db", "method"]
    R3["level"] = R3.level.astype(str)
    R3["gate_passed"] = R3.gate_passed.astype(bool)
    extra = {k: (c, "mean") for k, c in (("region_acc", "region_correct"), ("both_sources", "both_sources"),
                                         ("F1", "f1")) if c in R3}
    s3 = R3.groupby(keys).agg(n=("seed", "size"), gate_pass=("gate_passed", "mean"), err_mm=("peak_err_mm", "median"),
                              **extra).reset_index()
    s3 = s3.merge(R3[R3.gate_passed].groupby(keys).peak_err_mm.median().rename("err_mm_gated").reset_index(),
                  on=keys, how="left")
    print(f"Sensitivity analyses ({LABEL[R3_DATASET]} montage). err_mm: all recordings; err_mm_gated: gate passed.")
    show(save(s3, "R3_summary"), nd=2)

  R3 SNR 0.0: 560 rows (1002 s)
Sensitivity analyses (San Diego montage). err_mm: all recordings; err_mm_gated: gate passed.
               factor       level  snr_db method  n  gate_pass  err_mm  region_acc  both_sources   F1  err_mm_gated
        burst_freq_hz       13-17    -5.0   LCMV 10        1.0   35.84         0.7           NaN 0.24         35.84
        burst_freq_hz       13-17    -5.0   dSPM 10        1.0   37.62         0.4           NaN 0.24         37.62
        burst_freq_hz       13-17     0.0   LCMV 10        1.0   25.71         0.9           NaN 0.63         25.71
        burst_freq_hz       13-17     0.0   dSPM 10        1.0   10.15         1.0           NaN 0.63         10.15
        burst_freq_hz       16-24    -5.0   LCMV 10        0.9   18.78         0.7           NaN 0.36         17.13
        burst_freq_hz       16-24    -5.0   dSPM 10        0.9   23.23         0.5           NaN 0.36         23.09
        burst_freq_hz       16-24     0.0   LCMV 10        1.0 

## R7 — Random-effects table, bootstrap CIs, minimum detectable effects, equivalence

* Full random-effects (DerSimonian–Laird, Hartung–Knapp–Sidik–Jonkman CI) results for the manuscript's
  meta-analysis table, from the saved statistics of the main notebook.
* Percentile-bootstrap 95 % CIs (5 000 resamples) for every per-dataset effect size.
* Minimum detectable effect (80 % power, two-sided α = 0.05 and α = 0.01, the Holm threshold for the smallest of
  five p values), from the actual group sizes (t-test approximation; Mann–Whitney/Wilcoxon need ~5 % more).
* Two one-sided tests (TOST) for OFF vs ON with equivalence bounds ±`SESOI_DZ` d_z (paired t on the differences).

In [14]:
def mde_two(n1, n2, alpha, power=0.8):
    df = n1 + n2 - 2
    tc = stats.t.ppf(1 - alpha / 2, df)
    f = lambda d: stats.nct.sf(tc, df, d * np.sqrt(n1 * n2 / (n1 + n2))) - power
    return _root(f)


def _root(f):
    from scipy.optimize import brentq
    try:
        return float(brentq(f, 1e-4, 10))
    except ValueError:
        return np.nan


def mde_paired(n, alpha, power=0.8):
    df = n - 1
    tc = stats.t.ppf(1 - alpha / 2, df)
    f = lambda d: stats.nct.sf(tc, df, d * np.sqrt(n)) - power
    return _root(f)


if RUN["R7"]:
    rs = lambda k: pd.read_csv(WORK / "results" / f"stats_{k}.csv") if (WORK / "results" / f"stats_{k}.csv").exists() else pd.DataFrame()
    ph, ob, om = rs("pd_hc"), rs("offon_bursts"), rs("offon_maps")
    tabs = []
    if len(ph):
        tabs.append(meta(ph[ph.contrast == "PD-ON vs HC"], "g").assign(analysis="PD-ON vs HC (g)"))
    if len(ob):
        tabs.append(meta(ob, "d_z").assign(analysis="OFF vs ON bursts (d_z)"))
    if len(om):
        tabs.append(meta(om, "d_z").assign(analysis="OFF vs ON maps (d_z)"))
    MT = save(pd.concat(tabs, ignore_index=True), "R7_meta_full") if tabs else pd.DataFrame()
    print("Meta-analysis: fixed effect and random effects (DL τ², HKSJ 95 % CI):")
    show(MT, ["analysis", "metric", "k", "pooled", "ci_low", "ci_high", "p", "p_holm", "I2", "tau2", "RE_pooled",
              "RE_HKSJ_low", "RE_HKSJ_high", "RE_HKSJ_p"], nd=3)

    rng = np.random.default_rng(11)
    bt = []
    for ds in ANAT:
        r = per_rec(ds)
        if r is None:
            continue
        hc = r[r.condition == "hc"]
        for c in ("on", "off"):
            pdx = r[r.condition == c]
            if len(pdx) < 4 or len(hc) < 4:
                continue
            for m in R4_BURST:
                a_, b_ = pdx[m].dropna().values, hc[m].dropna().values
                g0, _ = hedges_g(a_, b_)
                bs = [hedges_g(rng.choice(a_, len(a_)), rng.choice(b_, len(b_)))[0] for _ in range(5000)]
                bt.append({"dataset": LABEL[ds], "contrast": f"PD-{c.upper()} vs HC", "metric": m, "effect": "g",
                           "estimate": g0, "ci_low": np.percentile(bs, 2.5), "ci_high": np.percentile(bs, 97.5),
                           "n1": len(a_), "n2": len(b_), "MDE_a05": mde_two(len(a_), len(b_), 0.05),
                           "MDE_a01": mde_two(len(a_), len(b_), 0.01)})
        off = r[r.condition == "off"].set_index("subject"); on = r[r.condition == "on"].set_index("subject")
        both = off.index.intersection(on.index)
        if len(both) > 5:
            for m in R4_BURST:
                dd = (off.loc[both, m] - on.loc[both, m]).dropna().values
                dz = dd.mean() / dd.std(ddof=1)
                bs = []
                for _ in range(5000):
                    x = rng.choice(dd, len(dd))
                    bs.append(x.mean() / x.std(ddof=1) if x.std(ddof=1) > 0 else np.nan)
                n = len(dd)
                se = dd.std(ddof=1) / np.sqrt(n)
                bound = SESOI_DZ * dd.std(ddof=1)
                p_lo = stats.t.sf((dd.mean() + bound) / se, n - 1)       # H0: mean ≤ −bound
                p_hi = stats.t.cdf((dd.mean() - bound) / se, n - 1)      # H0: mean ≥ +bound
                bt.append({"dataset": LABEL[ds], "contrast": "OFF vs ON", "metric": m, "effect": "d_z", "estimate": dz,
                           "ci_low": np.nanpercentile(bs, 2.5), "ci_high": np.nanpercentile(bs, 97.5), "n1": n, "n2": n,
                           "MDE_a05": mde_paired(n, 0.05), "MDE_a01": mde_paired(n, 0.01),
                           "TOST_p": max(p_lo, p_hi), "equivalent_at_SESOI": max(p_lo, p_hi) < 0.05})
    BT = save(pd.DataFrame(bt), "R7_bootstrap_mde_tost")
    print(f"\nPer-dataset effects with bootstrap 95 % CIs, minimum detectable effects (80 % power) and, for OFF vs ON, "
          f"TOST against ±{SESOI_DZ} d_z:")
    show(BT, nd=3)
    if len(BT):
        pooled = []
        for con, d in BT[BT.metric == "burst_amp"].groupby("contrast"):
            n1, n2 = d.n1.sum(), d.n2.sum()
            pooled.append({"contrast": con, "datasets": len(d), "n1_total": n1, "n2_total": n2,
                           "MDE_pooled_a05": mde_paired(int(n1), 0.05) if con == "OFF vs ON" else mde_two(int(n1), int(n2), 0.05)})
        print("\nMinimum detectable effect if the datasets were pooled (upper bound on meta-analytic sensitivity):")
        show(pd.DataFrame(pooled), nd=3)

    fl = rs("flow")
    if len(fl):
        print("\nDenominators for Table 8 (recordings per stage, full montage):")
        show(fl)

Meta-analysis: fixed effect and random effects (DL τ², HKSJ 95 % CI):
              analysis          metric  k  pooled  ci_low  ci_high     p  p_holm    I2  tau2  RE_pooled  RE_HKSJ_low  RE_HKSJ_high  RE_HKSJ_p
       PD-ON vs HC (g)         rate_hz  3  -0.334  -0.673    0.006 0.054   0.278 0.000 0.000     -0.334       -0.874         0.206      0.117
       PD-ON vs HC (g)          dur_ms  3  -0.263  -0.602    0.077 0.129   0.388 0.306 0.045     -0.217       -1.116         0.682      0.408
       PD-ON vs HC (g)   time_in_burst  3  -0.345  -0.685   -0.006 0.046   0.278 0.000 0.000     -0.345       -0.902         0.211      0.116
       PD-ON vs HC (g)          env_cv  3  -0.301  -0.643    0.042 0.085   0.341 0.719 0.266     -0.183       -1.566         1.199      0.626
       PD-ON vs HC (g)       burst_amp  3  -0.483  -0.827   -0.139 0.006   0.047 0.548 0.126     -0.399       -1.501         0.703      0.260
       PD-ON vs HC (g)      snr_est_db  3  -0.431  -0.776   -0.087 0.014   0.0